In [2]:
import json
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "dataset" / "kosovo_overall_biodiversity.csv"
PROCESSED_DIR = ROOT / "processed"
SOURCES_SUMMARY_PATH = PROCESSED_DIR / "sources_summary.csv"
GBIF_CACHE_PATH = PROCESSED_DIR / "gbif_datasets_cache.csv"
GBIF_API = "https://api.gbif.org/v1"
TOP_N = 15

PROCESSED_DIR.mkdir(exist_ok=True)


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

# 01b - Analiza e burimeve

CSV-ja nuk ka kolonat publishingOrgKey / publishingOrg. Publikuesin e marrim nga GBIF API:
`/dataset/{datasetKey}            -> titulli, DOI, publishingOrganizationKey`  
`/organization/{organizationKey}  -> emri dhe vendi i publikuesit`  
Rezultatet ruhen në cache (processed/gbif_datasets_cache.csv), që API të mos thirret në çdo ekzekutim.
Nëse API nuk përgjigjet për ndonjë dataset, publikuesi merret nga institutionCode.

## 1. LEXIMI I TË DHËNAVE

In [3]:
section("1. LEXIMI I TË DHËNAVE")
df = pd.read_csv(DATA_PATH, usecols=["datasetKey", "institutionCode", "basisOfRecord", "license"],
                 low_memory=False)
total = len(df)
print(f"Regjistrime: {total:,}  |  Dataset-e: {df.datasetKey.nunique()}  |  "
      f"institutionCode: {df.institutionCode.nunique()}")


1. LEXIMI I TË DHËNAVE
Regjistrime: 46,831  |  Dataset-e: 125  |  institutionCode: 108


## 2. METADATA NGA GBIF API

In [4]:
section("2. METADATA NGA GBIF API")


def get_json(url):
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            return json.load(r)
    except Exception as e:
        print(f"  ! {url}: {e}")
        return None


def fetch_dataset(key):
    d = get_json(f"{GBIF_API}/dataset/{key}")
    if d is None:
        return {"datasetKey": key}
    org_key = d.get("publishingOrganizationKey")
    org = get_json(f"{GBIF_API}/organization/{org_key}") if org_key else None
    return {
        "datasetKey": key,
        "datasetTitle": d.get("title"),
        "datasetDOI": d.get("doi"),
        "publishingOrgKey": org_key,
        "publishingOrg": org.get("title") if org else None,
        "publisherCountry": org.get("country") if org else None,
    }


keys = df.datasetKey.dropna().unique().tolist()
cache = pd.read_csv(GBIF_CACHE_PATH) if GBIF_CACHE_PATH.exists() else pd.DataFrame(columns=["datasetKey"])
cached_ok = cache.dropna(subset=["publishingOrg"]) if "publishingOrg" in cache else cache.iloc[0:0]
missing_keys = [k for k in keys if k not in set(cached_ok.datasetKey)]

print(f"Në cache: {len(keys) - len(missing_keys)}  |  Për t'u shkarkuar: {len(missing_keys)}")
if missing_keys:
    with ThreadPoolExecutor(max_workers=8) as pool:
        fetched = pd.DataFrame(list(pool.map(fetch_dataset, missing_keys)))
    cache = pd.concat([cached_ok, fetched], ignore_index=True)
    cache.to_csv(GBIF_CACHE_PATH, index=False)

meta = cache[cache.datasetKey.isin(keys)]
print(f"Publikues të gjetur nga API: {meta.publishingOrg.notna().sum()} / {len(keys)} dataset-e")


2. METADATA NGA GBIF API
Në cache: 125  |  Për t'u shkarkuar: 0
Publikues të gjetur nga API: 125 / 125 dataset-e


## 3. KOMBINIMI ME institutionCode

In [4]:
section("3. KOMBINIMI ME institutionCode")

# institutionCode-t për çdo dataset (një dataset mund të ketë disa)
inst = (df.dropna(subset=["institutionCode"])
          .groupby("datasetKey")["institutionCode"]
          .agg(lambda s: " | ".join(s.value_counts().index)))

df = df.merge(meta, on="datasetKey", how="left")
df["institutionCodes"] = df.datasetKey.map(inst)

# Fallback: nëse API nuk e ktheu publikuesin, përdoret institutionCode
df["publisher"] = df.publishingOrg
df["publisherSource"] = "GBIF API"
no_pub = df.publisher.isna()
df.loc[no_pub, "publisher"] = df.loc[no_pub, "institutionCode"].fillna("E panjohur")
df.loc[no_pub, "publisherSource"] = "institutionCode"
print(df.groupby("publisherSource").size().rename("regjistrime").to_string())


def share(counts):
    out = counts.rename("regjistrime").to_frame()
    out["perqindja"] = (out.regjistrime / total * 100).round(2)
    out["perqindja_kumulative"] = out.perqindja.cumsum().round(2)
    return out


3. KOMBINIMI ME institutionCode


publisherSource
GBIF API    46831


## 4. BURIMET KRYESORE SIPAS PUBLIKUESIT

In [5]:
section("4. BURIMET KRYESORE SIPAS PUBLIKUESIT")
by_pub = (df.groupby("publisher")
            .agg(regjistrime=("datasetKey", "size"),
                 dataset_e=("datasetKey", "nunique"),
                 basisOfRecord=("basisOfRecord", lambda s: s.mode().iat[0]),
                 license=("license", lambda s: ", ".join(s.value_counts().index)))
            .sort_values("regjistrime", ascending=False))
by_pub["perqindja"] = (by_pub.regjistrime / total * 100).round(2)
by_pub["perqindja_kumulative"] = by_pub.perqindja.cumsum().round(2)
print(f"Publikues gjithsej: {len(by_pub)}\n")
print(by_pub.head(TOP_N)[["regjistrime", "perqindja", "perqindja_kumulative", "dataset_e",
                          "basisOfRecord", "license"]])
print(f"\nTë tjerë ({len(by_pub) - TOP_N}): {by_pub.regjistrime.iloc[TOP_N:].sum():,} regjistrime "
      f"({by_pub.perqindja.iloc[TOP_N:].sum():.2f}%)")


4. BURIMET KRYESORE SIPAS PUBLIKUESIT


Publikues gjithsej: 80

                                                     regjistrime  perqindja  perqindja_kumulative  dataset_e        basisOfRecord                           license
publisher                                                                                                                                                          
Cornell Lab of Ornithology                                 26009      55.54                 55.54          1    HUMAN_OBSERVATION                         CC_BY_4_0
HabiProt                                                    8122      17.34                 72.88          2    HUMAN_OBSERVATION                      CC_BY_NC_4_0
iNaturalist.org                                             4286       9.15                 82.03          1    HUMAN_OBSERVATION  CC_BY_NC_4_0, CC_BY_4_0, CC0_1_0
Observation.org                                             1436       3.07                 85.10          1    HUMAN_OBSERVATION                      CC_BY

## 5. BURIMET KRYESORE SIPAS DATASET-IT

In [6]:
section("5. BURIMET KRYESORE SIPAS DATASET-IT")
by_ds = (df.groupby(["datasetKey", "datasetTitle", "publisher"], dropna=False)
           .size().sort_values(ascending=False))
by_ds = share(by_ds).reset_index()
print(by_ds.head(TOP_N)[["datasetTitle", "publisher", "regjistrime", "perqindja", "perqindja_kumulative"]]
      .to_string(index=False))


5. BURIMET KRYESORE SIPAS DATASET-IT
                                                                                                               datasetTitle                                           publisher  regjistrime  perqindja  perqindja_kumulative
                                                                                            EOD – eBird Observation Dataset                          Cornell Lab of Ornithology        26009      55.54                 55.54
                                                  Arthropod observations derived from HabiProt Alciphron database in Serbia                                            HabiProt         8107      17.31                 72.85
                                                                                    iNaturalist Research-grade Observations                                     iNaturalist.org         4286       9.15                 82.00
                                                                         O

## 6. SIPAS basisOfRecord

In [7]:
section("6. SIPAS basisOfRecord")
print(share(df.basisOfRecord.value_counts()))


6. SIPAS basisOfRecord
                     regjistrime  perqindja  perqindja_kumulative
basisOfRecord                                                    
HUMAN_OBSERVATION          42479      90.71                 90.71
PRESERVED_SPECIMEN          1748       3.73                 94.44
OCCURRENCE                  1269       2.71                 97.15
MATERIAL_SAMPLE              641       1.37                 98.52
MACHINE_OBSERVATION          376       0.80                 99.32
MATERIAL_CITATION            317       0.68                100.00
OBSERVATION                    1       0.00                100.00


## 7. SIPAS LICENCËS

In [8]:
section("7. SIPAS LICENCËS")
print(share(df.license.value_counts()))


7. SIPAS LICENCËS
              regjistrime  perqindja  perqindja_kumulative
license                                                   
CC_BY_4_0           31757      67.81                 67.81
CC_BY_NC_4_0        14267      30.46                 98.27
CC0_1_0               807       1.72                 99.99


## 8. PUBLIKUESIT KRYESORË × basisOfRecord DHE × LICENCA

In [9]:
section("8. PUBLIKUESIT KRYESORË × basisOfRecord DHE × LICENCA")
top_pubs = by_pub.index[:10]
top = df[df.publisher.isin(top_pubs)]
print(pd.crosstab(top.publisher, top.basisOfRecord, margins=True, margins_name="Totali")
        .reindex(list(top_pubs) + ["Totali"]))
print()
print(pd.crosstab(top.publisher, top.license, margins=True, margins_name="Totali")
        .reindex(list(top_pubs) + ["Totali"]))


8. PUBLIKUESIT KRYESORË × basisOfRecord DHE × LICENCA


basisOfRecord                                        HUMAN_OBSERVATION  OCCURRENCE  PRESERVED_SPECIMEN  Totali
publisher                                                                                                     
Cornell Lab of Ornithology                                       26009           0                   0   26009
HabiProt                                                          8122           0                   0    8122
iNaturalist.org                                                   4286           0                   0    4286
Observation.org                                                   1436           0                   0    1436
naturgucker.de                                                     721           0                   0     721
Pl@ntNet                                                           660           0                   0     660
University of Graz                                                   0           0                 659     659
V

license                                              CC0_1_0  CC_BY_4_0  CC_BY_NC_4_0  Totali
publisher                                                                                    
Cornell Lab of Ornithology                                 0      26009             0   26009
HabiProt                                                   0          0          8122    8122
iNaturalist.org                                          215        457          3614    4286
Observation.org                                            0          0          1436    1436
naturgucker.de                                             0        721             0     721
Pl@ntNet                                                   0        660             0     660
University of Graz                                         0        659             0     659
VectorNet                                                  0        587             0     587
Macedonian Ecological Society                              0

## 9. RUAJTJA: processed/sources_summary.csv

In [10]:
section("9. RUAJTJA: processed/sources_summary.csv")
group_cols = ["publisher", "publisherCountry", "publishingOrgKey", "datasetKey", "datasetTitle",
              "datasetDOI", "institutionCodes", "basisOfRecord", "license", "publisherSource"]
summary = (df.groupby(group_cols, dropna=False)
             .size().rename("regjistrime").reset_index()
             .sort_values("regjistrime", ascending=False))
summary["perqindja"] = (summary.regjistrime / total * 100).round(3)
summary.to_csv(SOURCES_SUMMARY_PATH, index=False)

assert summary.regjistrime.sum() == total
print(f"{len(summary)} rreshta (publikues × dataset × basisOfRecord × licencë) -> {SOURCES_SUMMARY_PATH}")


9. RUAJTJA: processed/sources_summary.csv


131 rreshta (publikues × dataset × basisOfRecord × licencë) -> C:\Master\PVDH\processed\sources_summary.csv


## 10. Vendim + arsyetim

### Tabela e burimeve kryesore

| # | Publikuesi | Vendi | Dataset-e | Regjistrime | % | % kumulative | basisOfRecord | Licenca |
|---|---|---|---|---|---|---|---|---|
| 1 | Cornell Lab of Ornithology (eBird) | US | 1 | 26,009 | 55.54 | 55.54 | HUMAN_OBSERVATION | CC BY 4.0 |
| 2 | HabiProt | RS | 2 | 8,122 | 17.34 | 72.88 | HUMAN_OBSERVATION | CC BY-NC 4.0 |
| 3 | iNaturalist.org | US | 1 | 4,286 | 9.15 | 82.03 | HUMAN_OBSERVATION | CC BY-NC / CC BY / CC0 |
| 4 | Observation.org | NL | 1 | 1,436 | 3.07 | 85.10 | HUMAN_OBSERVATION | CC BY-NC 4.0 |
| 5 | naturgucker.de | DE | 1 | 721 | 1.54 | 86.64 | HUMAN_OBSERVATION | CC BY 4.0 |
| 6 | Pl@ntNet | FR | 2 | 660 | 1.41 | 88.05 | HUMAN_OBSERVATION | CC BY 4.0 |
| 7 | University of Graz (Herbarium GZU) | AT | 1 | 659 | 1.41 | 89.46 | PRESERVED_SPECIMEN | CC BY 4.0 |
| 8 | VectorNet | SE | 3 | 587 | 1.25 | 90.71 | HUMAN_OBSERVATION | CC BY 4.0 |
| 9 | Macedonian Ecological Society | MK | 6 | 483 | 1.03 | 91.74 | OCCURRENCE | CC BY-NC 4.0 |
| 10 | CIAT | CO | 1 | 386 | 0.82 | 92.56 | OCCURRENCE | CC BY 4.0 |
| | *70 publikues të tjerë* | | 106 | 3,482 | 7.44 | 100.00 | | |
| | **Totali** | | **125** | **46,831** | **100.00** | | | |

Tabela e plotë (publikues × dataset × basisOfRecord × licencë, 131 rreshta) është në `processed/sources_summary.csv`.

### Vendimi

1. **Publikuesi merret nga GBIF API për të gjitha 125 dataset-et**, jo vetëm për ato kryesore. `institutionCode` mbetet vetëm si rezervë dhe ruhet në kolonën `institutionCodes` për gjurmueshmëri.
2. **Njësia kryesore e grupimit është publikuesi**, ndërsa `datasetKey` është nën-nivel. Për çdo grup raportohen numri dhe përqindja e regjistrimeve, `basisOfRecord` dhe licenca.
3. **Rezultati ruhet në `processed/sources_summary.csv`**, ndërsa përgjigjet e API-t ruhen në `processed/gbif_datasets_cache.csv`, që analiza të përsëritet pa thirrje të reja në API.

### Arsyetimi

- **Pse API dhe jo vetëm `institutionCode`:** `institutionCode` nuk e identifikon publikuesin në mënyrë të besueshme. Ka 108 vlera për 80 publikues, një publikues mund të ketë disa kode, dhe disa dataset-e e kanë bosh (p.sh. Observation.org, 1,436 regjistrime). API e gjeti publikuesin për 125/125 dataset-e, kështu që rezerva me `institutionCode` u përdor për 0 regjistrime.
- **Pse të gjitha dataset-et dhe jo vetëm kryesoret:** janë vetëm 125 thirrje dhe ruhen në cache. Me mbulim të plotë, rreshti "Të tjerë" mbetet i identifikuar dhe shuma e `sources_summary.csv` përputhet saktësisht me 46,831 regjistrimet (kontrollohet me `assert`).
- **Pse grupimi sipas publikuesit:** disa publikues kanë disa dataset-e (Macedonian Ecological Society 6, VectorNet 3, HabiProt dhe Pl@ntNet nga 2). Grupimi vetëm sipas `datasetKey` do ta copëtonte kontributin e të njëjtit burim.

### Çka tregon analiza dhe ndikimi në fazat e ardhshme

- **Përqendrim i lartë:** 3 publikues japin 82.03% të regjistrimeve dhe 10 publikues 92.56%. Vetëm eBird jep 55.54%, pra dataseti është i anuar nga zogjtë.
- **Kryesisht shkencë qytetare:** 90.71% e regjistrimeve janë `HUMAN_OBSERVATION`. Numri i vrojtimeve pasqyron përpjekjen e vëzhguesve, jo vetëm shpërndarjen reale të specieve. Kjo duhet pasur parasysh te agregimi dhe vizualizimi, p.sh. duke krahasuar rezultatet me dhe pa eBird ose duke stratifikuar sipas `kingdom`.
- **Licencat:** 67.81% CC BY 4.0, 30.46% CC BY-NC 4.0, 1.72% CC0. Për shkak të pjesës CC BY-NC, dataseti si tërësi përdoret vetëm për qëllime jokomerciale dhe citohet përmes DOI-t të shkarkimit nga GBIF.
- **Integrimi:** kolona `publisher` dhe `datasetKey` do të përdoren në `02_pastrimi.ipynb` për të kontrolluar nëse 3,336 dyfishimet e mundshme (takson + koordinata + data) vijnë nga burime të ndryshme.

### Dokumentimi i hapit

1. **Input:** `dataset/kosovo_overall_biodiversity.csv` (kolonat `datasetKey`, `institutionCode`, `basisOfRecord`, `license`) dhe GBIF API (`/v1/dataset/{datasetKey}`, `/v1/organization/{key}`).
2. **Çka u bë:** U mor publikuesi, titulli dhe DOI-ja për çdo dataset nga API, u kombinua me `institutionCode` dhe regjistrimet u grupuan sipas publikuesit, dataset-it, `basisOfRecord` dhe licencës.
3. **Rreshta para / rreshta pas:** Dataset-i nuk u ndryshua (46,831 rreshta). `sources_summary.csv` ka 131 rreshta dhe shuma e `regjistrime` është 46,831.
4. **Pse u bë:** CSV-ja nuk ka kolonat `publishingOrgKey` / `publishingOrg`. Pa to nuk dihet nga vijnë të dhënat, sa i besueshëm është secili burim dhe çfarë kufizimesh licence ka.
5. **Output:** `processed/sources_summary.csv`, `processed/gbif_datasets_cache.csv`, tabela e burimeve kryesore më lart.
6. **Commit:** Add source analysis decision and rationale.